# PMLS - ASSIGNMENT 1

## Model Deployment Using FastAPI

### Business Context:
A skin clinic conducted a marketing campaign targeting 10,000 customers. The objective is to analyse which customer segments are more likely to respond to the campaign based on demographics and purchase behaviour. Insights from this analysis will help improve future targeting strategies and increase campaign effectiveness.

|        Column Name        |                        Description                        |
|:-------------------------:|:---------------------------------------------------------:|
|          CustID           |            Unique identifier for each customer            |
|          Gender           |                    Gender of customer                     |
|         AgeGroup          |         Age bracket of customer (<30, 30-50, >50)         |
|   Purchase_Last_Quarter   | Whether customer made a purchase in last quarter (Yes/No) |
| Unique_Products_Purchased |   Number of unique products purchased in the last year    |
|   Response_to_Campaign    |      Target variable: Response to campaign (Yes/No)       |

In [1]:
import pandas as pd
import numpy as np

### Data Import and EDA

In [2]:
data = pd.read_csv("skin clinic campaign.csv")

data

,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes
...,...,...,...,...,...,...
9995,109996,Male,>50,Yes,13,Yes
9996,109997,Female,<30,No,5,No
9997,109998,Female,30-50,Yes,15,Yes
9998,109999,Male,30-50,No,2,No


In [3]:
data.dtypes

CustID                        int64
Gender                       object
AgeGroup                     object
Purchase_Last_Quarter        object
Unique_Products_Purchased     int64
Response_to_Campaign         object
dtype: object

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   CustID                     10000 non-null  int64 
 1   Gender                     10000 non-null  object
 2   AgeGroup                   10000 non-null  object
 3   Purchase_Last_Quarter      10000 non-null  object
 4   Unique_Products_Purchased  10000 non-null  int64 
 5   Response_to_Campaign       10000 non-null  object
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


In [5]:
data.nunique()

CustID                       10000
Gender                           2
AgeGroup                         3
Purchase_Last_Quarter            2
Unique_Products_Purchased       15
Response_to_Campaign             2
dtype: int64

In [6]:
data.isnull().sum()

CustID                       0
Gender                       0
AgeGroup                     0
Purchase_Last_Quarter        0
Unique_Products_Purchased    0
Response_to_Campaign         0
dtype: int64

In [7]:
data_copy = data.copy()

data_copy

,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes
...,...,...,...,...,...,...
9995,109996,Male,>50,Yes,13,Yes
9996,109997,Female,<30,No,5,No
9997,109998,Female,30-50,Yes,15,Yes
9998,109999,Male,30-50,No,2,No


### Response Rate for Customer Segments

In [8]:
def calculate_response_rate(data, column):
    results = []
    
    for category in data[column].unique():
        category_data = data[data[column] == category]
        
        total_customers = len(category_data)
        responders = (category_data["Response_to_Campaign"] == "Yes").sum()
        
        response_rate = (
            responders / total_customers * 100
            if total_customers > 0 else 0
        )
        
        results.append({
            "segment": category,
            "total_customers": int(total_customers),
            "responders": int(responders),
            "response_rate_pct": round(
                response_rate, 2
            )
        })
        
    return pd.DataFrame(results)

### Gender vs Campaign Response
Create a table showing the response rate (%) for Male and Female customers.

In [10]:
gender_table = calculate_response_rate(data_copy, "Gender")

gender_table

,segment,total_customers,responders,response_rate_pct
0,Female,6007,2629,43.77
1,Male,3993,1361,34.08


**Interpretation** - Female customers showed a higher response rate (43.77%) compared to male customers (34.08%).

### Age Group vs Campaign Response
Prepare a table showing the response rate (%) across the following age groups: (<30, 30-50, >50)

In [11]:
age_table = calculate_response_rate(data_copy, "AgeGroup")

age_table

,segment,total_customers,responders,response_rate_pct
0,30-50,4992,2349,47.06
1,>50,2509,818,32.60
2,<30,2499,823,32.93


**Interpretation** - Customers in the 30–50 age group had the highest response rate at 47.06%, while the <30 and >50 age groups had response rates of 32.93% and 32.60% respectively.

### Purchase in Last Quarter vs Campaign Response
Construct a table showing response rate (%) for customers based on whether they made a purchase in the last quarter (Yes/No).

In [12]:
purchase_last_quarter_table = calculate_response_rate(data_copy, "Purchase_Last_Quarter")

purchase_last_quarter_table

,segment,total_customers,responders,response_rate_pct
0,Yes,6514,3234,49.65
1,No,3486,756,21.69


**Interpretation** - Customers who made a purchase in the last quarter had a significantly higher response rate (49.65%) compared to those who did not make a purchase (21.69%).

### Product Usage vs Campaign Response
Categorise customers based on the number of unique products purchased in the last year: (1–4, 5–8, >8)

Create a table showing the response rate (%) for each category.

In [13]:
data_copy["Product_Usage"] = pd.cut(
    data_copy["Unique_Products_Purchased"],
    bins= [0, 4, 8, np.inf],
    labels= ["1-4", "5-8", ">8"]
)

data_copy["Product_Usage"].value_counts()

Product_Usage
>8     4206
5-8    3703
1-4    2091
Name: count, dtype: int64

In [14]:
product_usage_table = calculate_response_rate(data_copy, "Product_Usage")

product_usage_table

,segment,total_customers,responders,response_rate_pct
0,>8,4206,2187,52.00
1,5-8,3703,1425,38.48
2,1-4,2091,378,18.08


**Interpretation** - Customers who purchased more than 8 unique products had the highest response rate at 52.00%, followed by customers purchasing 5–8 products at 38.48%, and 1–4 products at 18.08%.